# Step 3: Flask Web Application
In this notebook, we will:
- Set up a Flask web app
- Load the trained model and label encoder
- Integrate the recommendation function
- Build HTML templates for input and output
- Test everything locally

**Goal:** A complete web application that predicts a disease from symptoms and provides full medical recommendations.

## 📁 Project Folder Structure
Before we write code, we have set up our Flask project directory like this:
```
MedicalRecommendationSystem/
├── app/
│   ├── templates/
│   │   ├── index.html
│   │   ├── result.html
│   ├── app.py
├── models/
│   ├── rf_model.pkl
│   ├── disease_encoder.pkl
├── data/
│   ├── description.csv, medications.csv, etc.
```


## 🔌 Install Flask
Installed Flask because we haven’t already:
```bash
pip install flask
```

## ✏️ `app.py`: Backend Code
This is our main Flask app code. It:
- Loads the model and encoder
- Handles form submission
- Predicts disease and shows recommendations

In [ ]:
from flask import Flask, request, render_template
import pickle
import pandas as pd
import numpy as np
import ast

# === Load Trained Model and Encoder ===
rf_model = pickle.load(open('./models/rf_model.pkl', 'rb'))
disease_encoder = pickle.load(open('./models/disease_encoder.pkl', 'rb'))

# === Load Supporting CSV Files ===
desc_df = pd.read_csv('./data/description.csv')
med_df = pd.read_csv('./data/medications.csv')
diet_df = pd.read_csv('./data/diets.csv')
prec_df = pd.read_csv('./data/precautions_df.csv')
work_df = pd.read_csv('./data/workout_df.csv')
training_df = pd.read_csv('./data/Training.csv')

# === Extract List of Symptoms from Training Data ===
symptom_list = list(training_df.columns[:-1])

# === Initialize Flask App ===
app = Flask(__name__)

# === Recommendation Function ===
def get_all_recommendations(disease_name):
    result = {}

    try:
        result['description'] = desc_df.loc[desc_df['Disease'] == disease_name, 'Description'].values[0]
    except:
        result['description'] = 'No description available.'

    try:
        meds = med_df.loc[med_df['Disease'] == disease_name, 'Medication'].values[0]
        result['medications'] = ast.literal_eval(meds)
    except:
        result['medications'] = []

    try:
        diet = diet_df.loc[diet_df['Disease'] == disease_name, 'Diet'].values[0]
        result['diet'] = ast.literal_eval(diet)
    except:
        result['diet'] = []

    try:
        prec_row = prec_df.loc[prec_df['Disease'] == disease_name].iloc[:, 2:].values.flatten()
        result['precautions'] = [p for p in prec_row if str(p) != 'nan']
    except:
        result['precautions'] = []

    try:
        workouts = work_df.loc[work_df['disease'] == disease_name, 'workout'].tolist()
        result['workouts'] = workouts
    except:
        result['workouts'] = []

    return result

# === Routes ===
@app.route('/')
def home():
    return render_template('index.html', symptoms=symptom_list)

@app.route('/predict', methods=['POST'])
def predict():
    # Get selected symptoms from form
    selected_symptoms = request.form.getlist('symptoms')

    # Convert to binary input vector
    input_vector = [1 if symptom in selected_symptoms else 0 for symptom in symptom_list]

    # Predict using model
    prediction = rf_model.predict([input_vector])[0]
    disease_name = disease_encoder.inverse_transform([prediction])[0]

    # Get recommendations
    recs = get_all_recommendations(disease_name)

    # Show result page
    return render_template('result.html', disease=disease_name, recs=recs)

# === Start Server ===
if __name__ == '__main__':
    app.run(debug=True)


## 🖼️ index.html

This page lets users select symptoms to get a diagnosis.

In [ ]:
<!-- File: templates/index.html -->
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Medical Recommendation System</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body {
            background-color: #f2f4f8;
            padding: 20px;
        }
        .container {
            background-color: #fff;
            padding: 40px;
            border-radius: 10px;
            box-shadow: 0 0 10px rgba(0, 0, 0, 0.1);
        }
        .form-check {
            margin-bottom: 10px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h2 class="mb-4 text-center">Select Your Symptoms</h2>
        <form action="/predict" method="POST">
            <div class="row">
                {% for symptom in symptoms %}
                <div class="col-md-4">
                    <div class="form-check">
                        <input class="form-check-input" type="checkbox" name="symptoms" value="{{ symptom }}" id="{{ symptom }}">
                        <label class="form-check-label" for="{{ symptom }}">{{ symptom.replace('_', ' ').title() }}</label>
                    </div>
                </div>
                {% endfor %}
            </div>
            <div class="text-center mt-4">
                <button type="submit" class="btn btn-primary btn-lg">Predict Disease</button>
            </div>
        </form>
    </div>
</body>
</html>


## 📄 result.html
This page displays the prediction and the recommendations.

In [ ]:
<!-- File: templates/result.html -->
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Diagnosis Result</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body {
            background-color: #f8f9fa;
            padding: 30px;
        }
        .container {
            background-color: #fff;
            padding: 30px;
            border-radius: 12px;
            box-shadow: 0 0 15px rgba(0,0,0,0.1);
        }
        h2, h3 {
            color: #003366;
        }
        ul {
            padding-left: 20px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h2 class="text-center mb-4">Predicted Disease: <strong class="text-danger">{{ disease }}</strong></h2>

        <h3>Description</h3>
        <p>{{ recs.description }}</p>

        <h3>Medications</h3>
        <ul>
            {% for med in recs.medications %}
                <li>{{ med }}</li>
            {% endfor %}
        </ul>

        <h3>Recommended Diet</h3>
        <ul>
            {% for food in recs.diet %}
                <li>{{ food }}</li>
            {% endfor %}
        </ul>

        <h3>Precautions</h3>
        <ul>
            {% for precaution in recs.precautions %}
                <li>{{ precaution }}</li>
            {% endfor %}
        </ul>

        <h3>Suggested Workouts</h3>
        <ul>
            {% for workout in recs.workouts %}
                <li>{{ workout }}</li>
            {% endfor %}
        </ul>

        <div class="text-center mt-4">
            <a href="/" class="btn btn-secondary">🔙 Back to Form</a>
        </div>
    </div>
</body>
</html>


## ✅ Next Step
- Run the app: `python app.py`
- Open `http://localhost:5000`
- Select symptoms, get prediction and recommendations!
